In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("Fashion_Retail_Sales.csv")

# Inspect the data
print("First 5 rows:")
print(df.head())

print("\nDataset information:")
print(df.info())

print("\nColumn names:")
print(df.columns.tolist())


First 5 rows:
   Customer Reference ID Item Purchased  Purchase Amount (USD) Date Purchase  \
0                   4018        Handbag                 4619.0    05-02-2023   
1                   4115          Tunic                 2456.0    11-07-2023   
2                   4019       Tank Top                 2102.0    23-03-2023   
3                   4097       Leggings                 3126.0    15-03-2023   
4                   3997         Wallet                 3003.0    27-11-2022   

   Review Rating Payment Method  
0            NaN    Credit Card  
1            2.0    Credit Card  
2            4.1           Cash  
3            3.2           Cash  
4            4.7           Cash  

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 3400 entries, 0 to 3399
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Customer Reference ID  3400 non-null   int64  
 1   Item Purchased         

In [2]:
# DATA QUALITY CHECK

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique customers:")
print(df["Customer Reference ID"].nunique())

print("\nTotal rows:")
print(len(df))

print("\nCustomer purchase counts:")
customer_counts = df["Customer Reference ID"].value_counts()
print(customer_counts.describe())

print("\nCustomers with more than one purchase:")
print((customer_counts > 1).sum())

print("\nPurchase amount summary:")
print(df["Purchase Amount (USD)"].describe())

print("\nReview rating summary:")
print(df["Review Rating"].describe())

print("\nPayment methods:")
print(df["Payment Method"].value_counts())

print("\nItems:")
print(df["Item Purchased"].value_counts())

Missing values:
Customer Reference ID      0
Item Purchased             0
Purchase Amount (USD)    650
Date Purchase              0
Review Rating            324
Payment Method             0
dtype: int64

Duplicate rows:
0

Unique customers:
166

Total rows:
3400

Customer purchase counts:
count    166.000000
mean      20.481928
std        4.375861
min       10.000000
25%       17.000000
50%       20.500000
75%       23.000000
max       34.000000
Name: count, dtype: float64

Customers with more than one purchase:
166

Purchase amount summary:
count    2750.000000
mean      156.709818
std       419.536669
min        10.000000
25%        57.000000
50%       110.000000
75%       155.750000
max      4932.000000
Name: Purchase Amount (USD), dtype: float64

Review rating summary:
count    3076.000000
mean        2.999057
std         1.156505
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Review Rating, dtype: float64

Payment met

In [3]:
# 2. DATA CLEANING & FEATURE ENGINEERING

clean_df = df.copy()

# Standardize column names

clean_df = clean_df.rename(columns={
    "Customer Reference ID": "customer_id",
    "Item Purchased": "item",
    "Purchase Amount (USD)": "purchase_amount",
    "Date Purchase": "purchase_date",
    "Review Rating": "review_rating",
    "Payment Method": "payment_method"
})

# Convert data types

# 1.Customer ID
clean_df["customer_id"] = clean_df["customer_id"].astype(int)

# 2.Purchase amount
clean_df["purchase_amount"] = pd.to_numeric(
    clean_df["purchase_amount"],
    errors="coerce"
)

# 3.Review rating
clean_df["review_rating"] = pd.to_numeric(
    clean_df["review_rating"],
    errors="coerce"
)

# 4.Purchase date
clean_df["purchase_date"] = pd.to_datetime(
    clean_df["purchase_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# 5.Check for invalid dates

print("Invalid dates:", clean_df["purchase_date"].isna().sum())

# 6.Add time features

clean_df["month"] = clean_df["purchase_date"].dt.to_period("M").astype(str)

clean_df["quarter"] = (
    clean_df["purchase_date"].dt.year.astype(str)
    + "-Q"
    + clean_df["purchase_date"].dt.quarter.astype(str)
)

clean_df["day_of_week"] = clean_df["purchase_date"].dt.day_name()


# 7.Create price bands

q1 = clean_df["purchase_amount"].quantile(0.25)
median = clean_df["purchase_amount"].quantile(0.50)
q3 = clean_df["purchase_amount"].quantile(0.75)

def price_band(amount):
    if pd.isna(amount):
        return "Unknown"
    elif amount <= q1:
        return "Low"
    elif amount <= median:
        return "Medium"
    elif amount <= q3:
        return "High"
    else:
        return "Premium"

clean_df["price_band"] = clean_df["purchase_amount"].apply(price_band)

# Remove exact duplicate rows

before = len(clean_df)

clean_df = clean_df.drop_duplicates()

after = len(clean_df)

print("Duplicate rows removed:", before - after)

# Final validation

print("\nFinal shape:")
print(clean_df.shape)

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nPrice band counts:")
print(clean_df["price_band"].value_counts(dropna=False))

print("\nDate range:")
print(clean_df["purchase_date"].min())
print(clean_df["purchase_date"].max())

print("\nPrice band boundaries:")
print(f"Low:      <= {q1:.2f}")
print(f"Medium:   {q1:.2f} - {median:.2f}")
print(f"High:     {median:.2f} - {q3:.2f}")
print(f"Premium:  > {q3:.2f}")

# Export cleaned dataset

clean_df.to_csv(
    "Fashion_Retail_Sales_Clean.csv",
    index=False
)

print("\nClean dataset exported successfully.")

Invalid dates: 0
Duplicate rows removed: 0

Final shape:
(3400, 10)

Missing values:
customer_id          0
item                 0
purchase_amount    650
purchase_date        0
review_rating      324
payment_method       0
month                0
quarter              0
day_of_week          0
price_band           0
dtype: int64

Price band counts:
price_band
Low        692
Medium     691
Premium    688
High       679
Unknown    650
Name: count, dtype: int64

Date range:
2022-10-02 00:00:00
2023-10-01 00:00:00

Price band boundaries:
Low:      <= 57.00
Medium:   57.00 - 110.00
High:     110.00 - 155.75
Premium:  > 155.75

Clean dataset exported successfully.


## Statistical Analysis

This section uses statistical methods to test whether observed patterns in the dataset are meaningful or could reasonably occur by chance.

The analysis includes:
- Descriptive statistics for purchase amount and review rating
- Correlation between purchase amount and review rating
- Credit Card vs Cash order-value hypothesis testing

In [9]:
print(df.columns.tolist())
print(df.head())

['Customer Reference ID', 'Item Purchased', 'Purchase Amount (USD)', 'Date Purchase', 'Review Rating', 'Payment Method']
   Customer Reference ID Item Purchased  Purchase Amount (USD) Date Purchase  \
0                   4018        Handbag                 4619.0    05-02-2023   
1                   4115          Tunic                 2456.0    11-07-2023   
2                   4019       Tank Top                 2102.0    23-03-2023   
3                   4097       Leggings                 3126.0    15-03-2023   
4                   3997         Wallet                 3003.0    27-11-2022   

   Review Rating Payment Method  
0            NaN    Credit Card  
1            2.0    Credit Card  
2            4.1           Cash  
3            3.2           Cash  
4            4.7           Cash  


In [10]:
import pandas as pd
import numpy as np
from scipy import stats

df_stats = df.copy()

# Rename columns for easier statistical analysis
df_stats = df_stats.rename(columns={
    "Customer Reference ID": "customer_id",
    "Item Purchased": "item",
    "Purchase Amount (USD)": "purchase_amount",
    "Date Purchase": "purchase_date",
    "Review Rating": "review_rating",
    "Payment Method": "payment_method"
})

# Convert date column
df_stats["purchase_date"] = pd.to_datetime(
    df_stats["purchase_date"],
    format="%d-%m-%Y"
)

print("Dataset shape:", df_stats.shape)
print("Purchase amount missing:", df_stats["purchase_amount"].isna().sum())
print("Review rating missing:", df_stats["review_rating"].isna().sum())

Dataset shape: (3400, 6)
Purchase amount missing: 650
Review rating missing: 324


### 1. Descriptive Statistics

Descriptive statistics summarize the central tendency, spread, and shape of purchase amounts and customer ratings.

In [11]:
amount = df_stats["purchase_amount"].dropna()
rating = df_stats["review_rating"].dropna()

descriptive_stats = pd.DataFrame({
    "Metric": ["Purchase Amount", "Review Rating"],
    "N": [amount.count(), rating.count()],
    "Mean": [amount.mean(), rating.mean()],
    "Median": [amount.median(), rating.median()],
    "Std Dev": [amount.std(), rating.std()],
    "Skewness": [amount.skew(), rating.skew()]
})

descriptive_stats.round(3)

,Metric,N,Mean,Median,Std Dev,Skewness
0,Purchase Amount,2750,156.710,110.0,419.537,8.497
1,Review Rating,3076,2.999,3.0,1.157,-0.015


In [12]:
correlation_data = df_stats[
    ["review_rating", "purchase_amount"]
].dropna()

pearson_r, p_value = stats.pearsonr(
    correlation_data["review_rating"],
    correlation_data["purchase_amount"]
)

print(f"Observations: {len(correlation_data)}")
print(f"Pearson correlation: {pearson_r:.4f}")
print(f"P-value: {p_value:.4f}")

Observations: 2487
Pearson correlation: 0.0451
P-value: 0.0244


In [13]:
credit_card = df_stats.loc[
    df_stats["payment_method"] == "Credit Card",
    "purchase_amount"
].dropna()

cash = df_stats.loc[
    df_stats["payment_method"] == "Cash",
    "purchase_amount"
].dropna()

credit_mean = credit_card.mean()
cash_mean = cash.mean()

t_stat, t_pvalue = stats.ttest_ind(
    credit_card,
    cash,
    equal_var=False
)

print(f"Credit Card transactions: {len(credit_card)}")
print(f"Cash transactions: {len(cash)}")
print(f"Credit Card average order value: ${credit_mean:.2f}")
print(f"Cash average order value: ${cash_mean:.2f}")
print(f"Welch's t-statistic: {t_stat:.4f}")
print(f"P-value: {t_pvalue:.4f}")

Credit Card transactions: 1438
Cash transactions: 1312
Credit Card average order value: $160.37
Cash average order value: $152.70
Welch's t-statistic: 0.4813
P-value: 0.6304


### Interpretation

Credit Card transactions had a slightly higher average order value than Cash transactions ($160.37 vs $152.70). However, the difference was not statistically significant (Welch's t-test, p = 0.6304).

Since the p-value is greater than 0.05, we fail to reject the null hypothesis. The data does not provide sufficient evidence that payment method is associated with a difference in average order value.

Mann–Whitney test

In [14]:
u_stat, mw_pvalue = stats.mannwhitneyu(
    credit_card,
    cash,
    alternative="two-sided"
)

print(f"Mann–Whitney U statistic: {u_stat:.2f}")
print(f"P-value: {mw_pvalue:.4f}")

Mann–Whitney U statistic: 930748.50
P-value: 0.5453


### 4. Non-Parametric Robustness Check

Because purchase amounts may not follow a normal distribution, a Mann–Whitney U test was used as a non-parametric robustness check.

The test produced a U statistic of 930,748.50 with a p-value of 0.5453.

Since p > 0.05, the result is not statistically significant. This supports the Welch's t-test conclusion that the observed difference in order value between Credit Card and Cash transactions is not statistically significant.

**Conclusion:** Payment method does not show a statistically significant difference in transaction value in this dataset.

### Overall Interpretation

Credit Card transactions had a higher observed average order value than Cash transactions ($160.37 vs $152.70). However, neither the Welch's t-test (p = 0.6304) nor the Mann–Whitney U test (p = 0.5453) found the difference to be statistically significant.

Therefore, the higher observed Credit Card average should not be interpreted as evidence that payment method affects transaction value.

In [15]:
amount = df_stats["purchase_amount"].dropna()
rating = df_stats["review_rating"].dropna()

descriptive_stats = pd.DataFrame({
    "Metric": ["Purchase Amount", "Review Rating"],
    "N": [amount.count(), rating.count()],
    "Mean": [amount.mean(), rating.mean()],
    "Median": [amount.median(), rating.median()],
    "Std Dev": [amount.std(), rating.std()],
    "Skewness": [amount.skew(), rating.skew()]
})

descriptive_stats.round(3)

,Metric,N,Mean,Median,Std Dev,Skewness
0,Purchase Amount,2750,156.710,110.0,419.537,8.497
1,Review Rating,3076,2.999,3.0,1.157,-0.015


### Interpretation

Purchase amounts were highly right-skewed (skewness = 8.497), with a mean of $156.71 compared with a median of $110.00. This indicates that a relatively small number of high-value transactions substantially influenced the average.

Review ratings were approximately symmetrically distributed, with a mean and median of about 3.0 and near-zero skewness (-0.015).

In [16]:
correlation_data = df_stats[
    ["review_rating", "purchase_amount"]
].dropna()

pearson_r, p_value = stats.pearsonr(
    correlation_data["review_rating"],
    correlation_data["purchase_amount"]
)

print(f"Observations: {len(correlation_data)}")
print(f"Pearson correlation: {pearson_r:.4f}")
print(f"P-value: {p_value:.4f}")

Observations: 2487
Pearson correlation: 0.0451
P-value: 0.0244


### Interpretation

There was a statistically significant but extremely weak positive correlation between review rating and purchase amount (Pearson r = 0.0451, p = 0.0244).

Since p < 0.05, the correlation is statistically significant at the 5% level. However, the correlation coefficient is very close to zero, indicating that the relationship is extremely weak and has limited practical significance.

This result should not be interpreted as evidence that higher ratings cause higher transaction values.

## Statistical Analysis Summary

| Analysis | Result | Significance | Interpretation |
|---|---|---|---|
| Purchase Amount Distribution | Mean = $156.71, Median = $110.00, Skewness = 8.497 | — | Highly right-skewed; high-value transactions influence the mean |
| Review Rating Distribution | Mean = 3.00, Median = 3.00, Skewness = -0.015 | — | Approximately symmetric distribution |
| Rating vs Purchase Amount | r = 0.0451 | Significant (p = 0.0244) | Extremely weak positive relationship with limited practical significance |
| Credit Card vs Cash AOV | $160.37 vs $152.70 | Not significant (p = 0.6304) | No sufficient evidence of a difference in average order value |
| Mann–Whitney Robustness Check | U = 930,748.50 | Not significant (p = 0.5453) | Supports the t-test conclusion |